# Module 12: Scaling & Distributed Training

## Learning Objectives
- Understand model scaling rules (depth, width)
- Data parallelism, tensor parallelism, pipeline parallelism
- Memory/performance tradeoffs
- Connect to Kyro's PipelineContext

## Concept: Parallelism Strategies

| Strategy | Description | Kyro Mapping |
|----------|-------------|---------------|
| Data Parallel | Split batch across GPUs | DistributedContext |
| Tensor Parallel | Split layers within GPU | DistributedContext::all_reduce |
| Pipeline Parallel | Split layers across GPUs | PipelineContext |

**Kyro mapping**: `src/distributed.rs`, `src/model/pipeline.rs`

In [ ]:
import math

def estimate_memory(vocab_size, hidden_dim, num_layers, seq_len, n_heads):
    param_count = (
        vocab_size * hidden_dim +
        num_layers * (4 * hidden_dim * hidden_dim) +
        num_layers * (hidden_dim * hidden_dim * 3) +
        num_layers * 2 * hidden_dim
    )
    activation_memory = seq_len * num_layers * hidden_dim * 2
    return param_count, activation_memory

configs = [
    ('Small', 1000, 128, 2, 16, 2),
    ('Medium', 5000, 512, 8, 32, 4),
    ('Large', 10000, 1024, 24, 16, 8),
]

for name, vocab, hidden, layers, heads, ctx in configs:
    params, mem = estimate_memory(vocab, hidden, layers, 2048, heads)
    print(f'{name}: {params/1e6:.1f}M params, {mem/1e6:.1f}MB activations')

print('\nIn Kyro: PipelineContext (src/model/pipeline.rs) splits layers across stages')

## Step 1: Pipeline Parallelism Simulation

Kyro's `PipelineContext` splits transformer layers across pipeline stages.

In [ ]:
class PipelineContext:
    def __init__(self, rank, world_size, total_layers):
        self.rank = rank
        self.world_size = world_size
        layers_per_gpu = (total_layers + world_size - 1) // world_size
        self.start_layer = rank * layers_per_gpu
        self.end_layer = min(self.start_layer + layers_per_gpu, total_layers)
    def is_first_stage(self):
        return self.rank == 0
    def is_last_stage(self):
        return self.rank == self.world_size - 1
    def should_process(self, idx):
        return self.start_layer <= idx < self.end_layer

for rank in range(4):
    ctx = PipelineContext(rank, 4, 32)
    print(f'Rank {rank}: layers [{ctx.start_layer}:{ctx.end_layer}], first={ctx.is_first_stage()}, last={ctx.is_last_stage()}')

## Exercises
1. Run a multi-GPU training job with torchrun.
2. Simulate pipeline parallelism with a toy cluster.
3. Explore `src/model/pipeline.rs::PipelineContext` and `src/distributed.rs`.